# Session 1: from a requirement to a size, by a rule
**A. Real-world problem.** A motor must turn a machine with a torque of 200 N m. What diameter should the steel shaft have?
A designer does not guess: there is a rule that links the torque, the strength of the steel and a safety margin to the smallest
safe diameter, and a second rule that rounds it up to a size that can really be bought. If the requirement changes, the rules give a new
part at once. How do such rules work, and how do we know the shaft is safe?

Educational model: a solid round shaft in pure torsion (twisting) and nothing else. Not a design code; real shafts also bend,
carry keyways and fatigue.

In [ ]:
import math
import pathlib

OUT = (pathlib.Path(__file__).resolve().parent if "__file__" in globals() else pathlib.Path(".")) / "out"
SHEAR_FRACTION = 0.577      # shear yield strength is about 0.577 times the tensile yield strength (von Mises)

## Step 1: the stress rule
When a solid round shaft of diameter d carries a torque T, the largest shear stress is at its surface:

    tau = 16 T / (pi d^3)          (T in N mm, d in mm, tau in MPa = N/mm^2)

The material may be loaded only to an allowable stress: the shear yield strength divided by a safety factor SF,
tau_allow = 0.577 sigma_y / SF. Setting tau = tau_allow and solving for d gives the smallest safe diameter:

    d_min = (16 T / (pi tau_allow))^(1/3)

In [ ]:
def allowable_shear(yield_mpa, safety_factor):
    if yield_mpa <= 0 or safety_factor < 1:
        raise ValueError("the yield strength must be positive and the safety factor at least 1")
    return SHEAR_FRACTION * yield_mpa / safety_factor


def shear_stress(torque_nm, diameter_mm):
    """Largest shear stress (MPa) at the surface of a solid round shaft."""
    if torque_nm < 0 or diameter_mm <= 0:
        raise ValueError("the torque must not be negative and the diameter must be positive")
    return 16 * torque_nm * 1000 / (math.pi * diameter_mm ** 3)


def min_diameter(torque_nm, yield_mpa, safety_factor):
    """Smallest diameter (mm) for which the shear stress stays within the allowable value."""
    if torque_nm <= 0:
        raise ValueError("the torque must be positive")
    tau = allowable_shear(yield_mpa, safety_factor)
    return (16 * torque_nm * 1000 / (math.pi * tau)) ** (1 / 3)

## Step 2: rounding up to a preferred size
Shafts are sold in standard sizes. The preferred-number series R10 spaces sizes by a constant factor of 10^(1/10) = 1.259, rounded
to convenient numbers: 1, 1.25, 1.6, 2, 2.5, 3.15, 4, 5, 6.3, 8, then the same times 10, 100 and so on. A second rule picks the
smallest standard size that is **not smaller** than d_min (rounding down would break the safety margin).

In [ ]:
R10 = [1.0, 1.25, 1.6, 2.0, 2.5, 3.15, 4.0, 5.0, 6.3, 8.0]


def preferred_sizes(lo=1.0, hi=1000.0):
    """All R10 sizes from lo to hi (mm)."""
    out, decade = [], 1.0
    while decade <= hi:
        out += [round(x * decade, 6) for x in R10 if lo <= x * decade <= hi]
        decade *= 10
    return out


def next_preferred(d_mm):
    """Smallest R10 size that is at least d_mm."""
    if d_mm <= 0:
        raise ValueError("the diameter must be positive")
    for size in preferred_sizes(0.1, 100000):
        if size >= d_mm * (1 - 1e-12):
            return size
    raise ValueError("no standard size is large enough")

## Step 3: the rule chain, and a check of what we got
Put the rules together: the requirement (torque, material, safety factor) gives d_min, the second rule gives the standard size, and a last step
computes what we really have: the stress in the chosen shaft and the safety factor it gives, which is a little above the one we asked for.
For a motor we are usually given power P (kW) and speed n (rpm) instead of the torque: T = 9549 P / n (N m).

In [ ]:
def torque_from_power(power_kw, rpm):
    if power_kw <= 0 or rpm <= 0:
        raise ValueError("the power and the speed must be positive")
    return 9549.0 * power_kw / rpm


def design_shaft(torque_nm, yield_mpa=350.0, safety_factor=2.5):
    """Apply the rules: d_min, then the standard size, then check. Returns a dictionary."""
    d_min = min_diameter(torque_nm, yield_mpa, safety_factor)
    d = next_preferred(d_min)
    tau = shear_stress(torque_nm, d)
    return {"torque": torque_nm, "d_min": d_min, "d": d, "tau": tau,
            "sf_actual": SHEAR_FRACTION * yield_mpa / tau, "margin": d / d_min - 1,
            "mass_per_m": 7850 * math.pi / 4 * (d / 1000) ** 2}

In [ ]:
if __name__ == "__main__":
    import matplotlib.pyplot as plt

    OUT.mkdir(exist_ok=True)
    s = design_shaft(200.0)
    print(f"T = 200 N m, yield 350 MPa, SF 2.5: tau_allow = {allowable_shear(350, 2.5):.1f} MPa, d_min = {s['d_min']:.2f} mm, standard size {s['d']:g} mm")
    print(f"actual stress {s['tau']:.1f} MPa, actual safety factor {s['sf_actual']:.2f}, {s['margin'] * 100:.0f} percent above the minimum, {s['mass_per_m']:.2f} kg per metre")
    t15 = torque_from_power(15, 1450)
    print(f"15 kW at 1450 rpm: T = {t15:.1f} N m, shaft {design_shaft(t15)['d']:g} mm")
    print("R10 sizes from 10 to 100 mm:", preferred_sizes(10, 100))

    ts = [20 * k for k in range(1, 101)]
    fig, ax = plt.subplots(figsize=(6.4, 4.2))
    ax.plot(ts, [min_diameter(t, 350, 2.5) for t in ts], color="tab:blue", label="smallest safe diameter (rule 1)")
    ax.step(ts, [next_preferred(min_diameter(t, 350, 2.5)) for t in ts], where="post", color="tab:red", label="standard size (rule 2)")
    ax.set_xlabel("torque T (N m)")
    ax.set_ylabel("shaft diameter (mm)")
    ax.set_title("Two rules in a row: minimum diameter, then standard size")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    fig.tight_layout()
    fig.savefig(OUT / "diameter_vs_torque.png", dpi=120)

    fig2, ax2 = plt.subplots(figsize=(6.4, 4.2))
    for sf, col in ((1.5, "tab:green"), (2.5, "tab:blue"), (4.0, "tab:red")):
        ax2.plot(ts, [design_shaft(t, 350, sf)["sf_actual"] for t in ts], color=col, label=f"asked for SF = {sf}")
    ax2.set_xlabel("torque T (N m)")
    ax2.set_ylabel("safety factor of the standard shaft")
    ax2.set_title("The standard size always gives at least the safety factor asked for")
    ax2.grid(alpha=0.3)
    ax2.legend(fontsize=8)
    fig2.tight_layout()
    fig2.savefig(OUT / "safety_factor.png", dpi=120)

## Where this stops
One rule decides one number. A real part has many numbers that depend on each other, and the right answer for one rule (strength) can be wrong for another (stiffness).
Session 2 puts two competing rules on one part and lets the stricter one decide.

## Checkpoint
Run this last cell. It prints PASS when your work is correct.

In [ ]:
"""Checkpoint for Session 1. Prints PASS or FAIL."""
import numpy as np


def run_checks(L):
    """L is the lesson module (or notebook namespace). Returns a list of (label, passed)."""
    checks = []
    tau = L.allowable_shear(350, 2.5)
    checks.append((f"worked example: allowable shear stress 80.8 MPa (got {tau:.1f})", abs(tau - 80.78) < 0.01))
    s = L.design_shaft(200.0)
    checks.append((f"worked example: minimum diameter 23.28 mm (got {s['d_min']:.2f})", abs(s["d_min"] - 23.28) < 0.01))
    checks.append((f"worked example: standard size 25 mm (got {s['d']:g})", s["d"] == 25.0))
    checks.append((f"worked example: stress in the 25 mm shaft is 65.2 MPa (got {s['tau']:.1f})", abs(s["tau"] - 65.19) < 0.05))
    checks.append((f"worked example: actual safety factor 3.10 (got {s['sf_actual']:.2f})", abs(s["sf_actual"] - 3.098) < 0.005))
    checks.append((f"worked example: 15 kW at 1450 rpm is 98.8 N m (got {L.torque_from_power(15, 1450):.1f})", abs(L.torque_from_power(15, 1450) - 98.78) < 0.01))
    ok = True
    for t, sy, sf in ((50, 250, 1.5), (200, 350, 2.5), (1000, 600, 4.0), (5, 200, 3.0)):
        d = L.min_diameter(t, sy, sf)
        ok &= abs(L.shear_stress(t, d) - L.allowable_shear(sy, sf)) < 1e-9 * L.allowable_shear(sy, sf)
    checks.append(("validation: the stress in a shaft of exactly d_min equals the allowable stress (the rule inverts the stress formula)", ok))
    checks.append(("validation: doubling the torque multiplies d_min by 2^(1/3) = 1.26", abs(L.min_diameter(400, 350, 2.5) / L.min_diameter(200, 350, 2.5) - 2 ** (1 / 3)) < 1e-12))
    expected = [10.0, 12.5, 16.0, 20.0, 25.0, 31.5, 40.0, 50.0, 63.0, 80.0, 100.0]
    checks.append(("the R10 sizes from 10 to 100 mm are 10, 12.5, 16, 20, 25, 31.5, 40, 50, 63, 80, 100", L.preferred_sizes(10, 100) == expected))
    sizes = L.preferred_sizes(1, 1000)
    ratios = [b / a for a, b in zip(sizes, sizes[1:])]
    checks.append(("validation: neighbouring R10 sizes differ by a factor of about 1.26 (between 1.18 and 1.30)", all(1.18 < r < 1.30 for r in ratios)))
    checks.append(("a size that is exactly on the list stays: next_preferred(25) = 25 and next_preferred(25.01) = 31.5", L.next_preferred(25.0) == 25.0 and L.next_preferred(25.01) == 31.5))
    checks.append(("rounding goes up, not to the nearest: d_min = 20.4 gives 25 mm although 20 is nearer", L.next_preferred(20.4) == 25.0))
    ok = True
    for t in range(10, 1500, 37):
        r = L.design_shaft(float(t))
        ok &= r["d"] >= r["d_min"] and r["sf_actual"] >= 2.5 - 1e-9 and r["margin"] < 0.27
    checks.append(("for 40 torques, the standard shaft is never smaller than d_min, its safety factor is at least the one asked for and it is less than 27 percent oversize", ok))
    checks.append(("a stronger steel needs a smaller shaft", L.min_diameter(200, 500, 2.5) < L.min_diameter(200, 350, 2.5)))
    for bad, label in ((lambda: L.min_diameter(0, 350, 2.5), "zero torque"), (lambda: L.min_diameter(100, 350, 0.9), "a safety factor below 1"),
                       (lambda: L.shear_stress(100, 0), "a zero diameter"), (lambda: L.next_preferred(-5), "a negative diameter"),
                       (lambda: L.torque_from_power(15, 0), "a speed of zero"), (lambda: L.allowable_shear(-1, 2), "a negative yield strength")):
        try:
            bad()
            raised = False
        except ValueError:
            raised = True
        checks.append((f"edge case: {label} gives an error", raised))
    return checks
import types
L = types.SimpleNamespace(**{k: v for k, v in globals().items() if not k.startswith('_')})
ok = True
for label, passed in run_checks(L):
    print(('PASS  ' if passed else 'FAIL  ') + label)
    ok &= bool(passed)
print('PASS' if ok else 'FAIL')